# Data Quality Validation for GEV Dependency Distance Analysis

This notebook demonstrates six data quality validation analyses for GEV dependency distance research using pre-computed results from 238 UD treebanks (420K sentences) and 106 Grambank-UD overlapping languages.

**Analyses performed:**
1. **Grambank-UD cross-validation** - Correlating UD morphological richness with Grambank morph index
2. **Entropy threshold sensitivity** - Checking stability of word-order entropy at thresholds 10 vs 20
3. **Representativeness** - Coverage of exact-length subset vs full corpus
4. **Autocorrelation diagnostics** - Ljung-Box tests on treebank-bin combinations
5. **Annotation confound** - Testing if feat_completeness confounds morph_richness
6. **Family-level bias** - Checking distribution balance across languages

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# All packages used are pre-installed on Colab; install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')
    # scipy 1.16.3 requires Python >=3.11 (Colab); use 1.15.3 for Python 3.10
    try:
        _pip('scipy==1.16.3')
    except Exception:
        _pip('scipy>=1.13.0,<2')
    _pip('statsmodels==0.14.6')

In [ ]:
import json
import os
import math
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from scipy import stats as sp_stats
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-42dac1-word-order-entropy-predicts-ordinal-tail/main/experiment_iter2_data_quality_va/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded {len(data['datasets'][0]['examples'])} treebank examples")
print(f"Global summary keys: {list(data['metadata']['global_summary'].keys())}")

In [ ]:
# ============================================================
# CONFIG: Tunable parameters (set to minimum for demo)
# ============================================================
N_BOOTSTRAP = 100          # Original: 1000 — bootstrap resamples for CIs
CONFOUND_THRESHOLD = 0.7   # rho threshold for annotation confound
LOW_FC_THRESHOLD = 0.3     # feat_completeness threshold for outliers
LOW_COVERAGE_THRESHOLD = 0.10  # coverage threshold for representativeness
AUTOCORR_CONCERN_THRESHOLD = 0.20  # proportion threshold for autocorrelation concern
RANDOM_SEED = 42

## Parse Data Structure

Each treebank example contains metadata fields and two prediction fields:
- `predict_comprehensive_validation`: Results from all 6 analyses (Grambank, entropy, representativeness, autocorrelation, annotation confound, quality flags)
- `predict_baseline_range_check`: Simple plausibility checks verifying numeric fields fall within expected ranges

In [ ]:
# Parse all examples into a DataFrame with expanded JSON fields
examples = data["datasets"][0]["examples"]
global_summary = data["metadata"]["global_summary"]

records = []
for ex in examples:
    output = json.loads(ex["output"])
    comprehensive = json.loads(ex["predict_comprehensive_validation"])
    baseline = json.loads(ex["predict_baseline_range_check"])

    rec = {
        "treebank_id": ex["input"],
        "language": ex["metadata_language"],
        "iso_code": ex["metadata_iso_code"],
        "modality": ex["metadata_modality"],
        "genre": ex["metadata_genre"],
        "morph_richness": ex["metadata_morph_richness"],
        "word_order_entropy": ex["metadata_word_order_entropy"],
        "feat_completeness": ex["metadata_feat_completeness"],
        "n_quality_flags": ex["metadata_n_quality_flags"],
        # From output
        "mean_dd_all": output["mean_dd_all"],
        "n_sentences_total": output["n_sentences_total"],
        "n_binned": output["n_binned"],
        "coverage": output["coverage"],
        # From comprehensive validation
        "overall_quality": comprehensive["overall_quality"],
        "grambank_in_overlap": comprehensive["grambank"]["in_overlap"],
        "has_autocorrelation": comprehensive["autocorrelation"]["has_autocorrelation"],
        "n_bins_tested": comprehensive["autocorrelation"]["n_bins_tested"],
        "n_bins_significant": comprehensive["autocorrelation"]["n_bins_significant"],
        "fc_is_low": comprehensive["annotation_confound"]["is_low"],
        "low_coverage": comprehensive["representativeness"]["low_coverage"],
        # Baseline
        "baseline_pass": baseline["overall_quality"] == "PASS",
    }

    # Grambank data (if in overlap)
    if comprehensive["grambank"].get("in_overlap"):
        rec["ud_morph_richness"] = comprehensive["grambank"].get("ud_morph_richness")
        rec["grambank_morph_index"] = comprehensive["grambank"].get("grambank_morph_index")

    # Entropy data (if computed)
    ent = comprehensive.get("entropy", {})
    if "entropy_threshold_10" in ent:
        rec["entropy_threshold_10"] = ent["entropy_threshold_10"]
        rec["entropy_threshold_20"] = ent["entropy_threshold_20"]
        rec["entropy_abs_diff"] = ent.get("abs_diff", abs(ent["entropy_threshold_10"] - ent["entropy_threshold_20"]))

    records.append(rec)

df = pd.DataFrame(records)
print(f"Parsed {len(df)} treebanks into DataFrame")
print(f"\nQuality distribution:")
print(df["overall_quality"].value_counts().to_string())
print(f"\nSample row:")
print(df.iloc[0].to_string())

## Analysis 1: Grambank-UD Morphological Richness Cross-Validation

Cross-validate UD morphological richness against Grambank morph index for languages in the overlap set. Compute Spearman/Pearson correlations with bootstrap confidence intervals.

In [ ]:
# Analysis 1: Grambank Cross-Validation
# Filter to treebanks with Grambank overlap
gb_df = df[df["grambank_in_overlap"] & df["ud_morph_richness"].notna()].copy()
n_overlap = len(gb_df)
print(f"Grambank-UD overlap treebanks in demo: {n_overlap}")

if n_overlap >= 5:
    # Compute correlations
    spearman_r, spearman_p = sp_stats.spearmanr(gb_df["ud_morph_richness"], gb_df["grambank_morph_index"])
    pearson_r, pearson_p = sp_stats.pearsonr(gb_df["ud_morph_richness"], gb_df["grambank_morph_index"])

    # Bootstrap 95% CIs
    rng = np.random.default_rng(RANDOM_SEED)
    boot_spearman, boot_pearson = [], []
    for _ in range(N_BOOTSTRAP):
        idx = rng.choice(n_overlap, size=n_overlap, replace=True)
        sample = gb_df.iloc[idx]
        sr, _ = sp_stats.spearmanr(sample["ud_morph_richness"], sample["grambank_morph_index"])
        pr, _ = sp_stats.pearsonr(sample["ud_morph_richness"], sample["grambank_morph_index"])
        boot_spearman.append(float(sr))
        boot_pearson.append(float(pr))

    spearman_ci = (float(np.percentile(boot_spearman, 2.5)), float(np.percentile(boot_spearman, 97.5)))
    pearson_ci = (float(np.percentile(boot_pearson, 2.5)), float(np.percentile(boot_pearson, 97.5)))

    # Outlier detection: feat_completeness < LOW_FC_THRESHOLD
    outliers = gb_df[gb_df["feat_completeness"] < LOW_FC_THRESHOLD]

    interpretation = (
        "STRONG" if abs(spearman_r) > 0.6 else
        "MODERATE" if abs(spearman_r) > 0.4 else
        "WEAK" if abs(spearman_r) > 0.2 else "NEGLIGIBLE"
    )

    print(f"\nSpearman rho = {spearman_r:.4f} (p = {spearman_p:.4e}), 95% CI: [{spearman_ci[0]:.4f}, {spearman_ci[1]:.4f}]")
    print(f"Pearson r   = {pearson_r:.4f} (p = {pearson_p:.4e}), 95% CI: [{pearson_ci[0]:.4f}, {pearson_ci[1]:.4f}]")
    print(f"Interpretation: {interpretation}")
    print(f"Outliers (feat_completeness < {LOW_FC_THRESHOLD}): {len(outliers)}")
    print(f"\nFull-dataset result (238 treebanks): Spearman rho = {global_summary['grambank_spearman_rho']}")
else:
    print("Too few overlap treebanks for correlation analysis")

## Analysis 5: Annotation-Completeness Confound

Test whether `feat_completeness` confounds `morph_richness` (Spearman rho > 0.7 indicates a significant confound). Also check the correlation of `morph_richness` vs `mean_dd_all`.

In [ ]:
# Analysis 5: Annotation Confound
fc = df["feat_completeness"].values
mr = df["morph_richness"].values

rho_fc_mr, p_fc_mr = sp_stats.spearmanr(fc, mr)
is_confound = abs(rho_fc_mr) > CONFOUND_THRESHOLD

# Additional: morph_richness vs mean_dd_all
rho_mr_dd, p_mr_dd = sp_stats.spearmanr(mr, df["mean_dd_all"].values)

# Low feat_completeness treebanks
low_fc = df[df["feat_completeness"] < LOW_FC_THRESHOLD][["treebank_id", "language", "feat_completeness", "morph_richness"]]

print(f"Annotation Confound Assessment (n={len(df)} treebanks):")
print(f"  feat_completeness vs morph_richness: rho = {rho_fc_mr:.4f} (p = {p_fc_mr:.4e})")
print(f"  Is confound (|rho| > {CONFOUND_THRESHOLD}): {is_confound}")
print(f"  morph_richness vs mean_dd_all: rho = {rho_mr_dd:.4f} (p = {p_mr_dd:.4e})")
print(f"  Treebanks with feat_completeness < {LOW_FC_THRESHOLD}: {len(low_fc)}")
if len(low_fc) > 0:
    print(low_fc.to_string(index=False))
print(f"\nFull-dataset result: confound_rho = {global_summary['confound_rho']}, is_confound = {global_summary['is_confound']}")
if is_confound:
    print("WARNING: feat_completeness strongly confounds morph_richness. Include feat_completeness as control in regression.")
else:
    print("feat_completeness is not a dominant confound, but still include as control.")

## Analysis 3: Representativeness of Exact-Length Subset

Assess whether the binned (exact-length) subset is representative of the full corpus. Check coverage statistics, correlation with syntactic complexity, and modality/genre breakdowns.

In [ ]:
# Analysis 3: Representativeness
print(f"Representativeness (n={len(df)} treebanks):")
print(f"  Mean coverage:   {df['coverage'].mean():.4f}")
print(f"  Median coverage: {df['coverage'].median():.4f}")
print(f"  Min coverage:    {df['coverage'].min():.4f}")
print(f"  Max coverage:    {df['coverage'].max():.4f}")
print(f"  Std coverage:    {df['coverage'].std():.4f}")
print(f"  Total binned:    {int(df['n_binned'].sum()):,}")
print(f"  Total all:       {int(df['n_sentences_total'].sum()):,}")
overall_cov = df["n_binned"].sum() / df["n_sentences_total"].sum()
print(f"  Overall coverage: {overall_cov:.4f}")

# Correlation: coverage vs mean_dd_all
rho_dd, p_dd = sp_stats.spearmanr(df["coverage"], df["mean_dd_all"])
print(f"\nCoverage vs mean_dd_all: rho = {rho_dd:.4f} (p = {p_dd:.4e})")
if rho_dd < -0.2 and p_dd < 0.05:
    print("  => Higher mean DD treebanks have LOWER coverage in bins")
else:
    print("  => No strong coverage bias by syntactic complexity")

# Modality comparison
spoken = df[df["modality"] == "spoken"]["coverage"]
written = df[df["modality"] == "written"]["coverage"]
if len(spoken) >= 2 and len(written) >= 2:
    print(f"\nSpoken coverage:  mean={spoken.mean():.4f} (n={len(spoken)})")
    print(f"Written coverage: mean={written.mean():.4f} (n={len(written)})")

# Genre breakdown
print("\nGenre breakdown:")
for genre, grp in df.groupby("genre"):
    print(f"  {genre}: mean={grp['coverage'].mean():.4f}, median={grp['coverage'].median():.4f}, n={len(grp)}")

# Low-coverage treebanks
low_cov = df[df["coverage"] < LOW_COVERAGE_THRESHOLD].sort_values("coverage")
print(f"\nLow-coverage treebanks (< {LOW_COVERAGE_THRESHOLD}): {len(low_cov)}")
if len(low_cov) > 0:
    print(low_cov[["treebank_id", "language", "coverage", "n_sentences_total"]].to_string(index=False))
print(f"\nFull-dataset overall coverage: {global_summary['overall_coverage']}")

## Analysis 4: Autocorrelation Diagnostics

Summarize Ljung-Box serial dependence test results across treebank-bin combinations. If >20% show significant autocorrelation, block subsampling is recommended for GEV fitting.

In [ ]:
# Analysis 4: Autocorrelation Summary
n_tested = int(df["n_bins_tested"].sum())
n_significant = int(df["n_bins_significant"].sum())
prop_sig = n_significant / n_tested if n_tested > 0 else 0.0
n_tb_with_autocorr = int(df["has_autocorrelation"].sum())

print(f"Autocorrelation Diagnostics (demo subset):")
print(f"  Treebank-bin combinations tested: {n_tested}")
print(f"  Significant at p<0.05: {n_significant} ({prop_sig:.1%})")
print(f"  Treebanks with any autocorrelation: {n_tb_with_autocorr}/{len(df)}")
print(f"  Recommend block subsampling: {prop_sig > AUTOCORR_CONCERN_THRESHOLD}")

if prop_sig > AUTOCORR_CONCERN_THRESHOLD:
    print(f"\n  CONCERN: >{AUTOCORR_CONCERN_THRESHOLD:.0%} of bin-treebank combinations show significant serial")
    print("  dependence. Block-of-blocks subsampling recommended for GEV fitting.")
else:
    print(f"\n  ACCEPTABLE: {prop_sig:.1%} show dependence, within expected false-positive range.")

# Show per-treebank detail
autocorr_detail = df[df["has_autocorrelation"]][["treebank_id", "language", "n_bins_tested", "n_bins_significant"]]
if len(autocorr_detail) > 0:
    print(f"\nTreebanks with detected autocorrelation:")
    print(autocorr_detail.to_string(index=False))

print(f"\nFull-dataset proportion significant: {global_summary['autocorr_proportion_significant']}")

## Analysis 2: Entropy Threshold Sensitivity

Compare word-order entropy computed at threshold=10 vs threshold=20. High Spearman correlation (>0.95) indicates robustness to threshold choice.

In [ ]:
# Analysis 2: Entropy Threshold Sensitivity
ent_df = df[df["entropy_threshold_10"].notna()].copy()
n_ent = len(ent_df)
print(f"Treebanks with entropy data: {n_ent}")

if n_ent >= 5:
    ent10 = ent_df["entropy_threshold_10"].values
    ent20 = ent_df["entropy_threshold_20"].values

    spearman_10v20, p_10v20 = sp_stats.spearmanr(ent10, ent20)
    mean_abs_diff = float(np.mean(np.abs(ent10 - ent20)))

    # Rank shifts
    rank10 = sp_stats.rankdata(ent10)
    rank20 = sp_stats.rankdata(ent20)
    rank_shifts = np.abs(rank10 - rank20)
    n_big_shift = int(np.sum(rank_shifts > 5))

    is_robust = float(spearman_10v20) > 0.95

    print(f"\nSpearman rho (threshold 10 vs 20): {spearman_10v20:.4f} (p = {p_10v20:.4e})")
    print(f"Is robust (rho > 0.95): {is_robust}")
    print(f"Mean absolute difference: {mean_abs_diff:.6f}")
    print(f"Treebanks with rank shift > 5: {n_big_shift}")
    print(f"\nPer-treebank entropy values:")
    print(ent_df[["treebank_id", "language", "entropy_threshold_10", "entropy_threshold_20", "entropy_abs_diff"]].to_string(index=False))
    print(f"\nFull-dataset result: Spearman rho = {global_summary['entropy_spearman_10v20']}, robust = {global_summary['entropy_is_robust']}")
else:
    print("Too few treebanks with entropy data for analysis")

## Analysis 6: Family-Level Bias and Distribution Check

Check language distribution balance and test normality of key variables (morph_richness, word_order_entropy) via Shapiro-Wilk test.

In [ ]:
# Analysis 6: Family Bias and Distribution Check
lang_counts = df["language"].value_counts()
multi_tb_languages = lang_counts[lang_counts > 1]

print(f"Family/Language Bias Check (n={len(df)} treebanks):")
print(f"  Unique languages: {df['language'].nunique()}")
print(f"  Treebanks per language: mean={lang_counts.mean():.2f}, median={lang_counts.median():.1f}, max={lang_counts.max()}")
print(f"  Languages with multiple treebanks: {len(multi_tb_languages)}")
if len(multi_tb_languages) > 0:
    print(f"    {dict(multi_tb_languages)}")

# Normality tests
morph_values = df["morph_richness"].values
wo_values = df["word_order_entropy"].values

if len(morph_values) >= 3:
    shapiro_mr, shapiro_mr_p = sp_stats.shapiro(morph_values)
    shapiro_wo, shapiro_wo_p = sp_stats.shapiro(wo_values)

    print(f"\nMorph richness distribution:")
    print(f"  mean={np.mean(morph_values):.4f}, std={np.std(morph_values):.4f}")
    print(f"  min={morph_values.min():.4f}, max={morph_values.max():.4f}")
    print(f"  Shapiro-Wilk W={shapiro_mr:.4f}, p={shapiro_mr_p:.4e}, normal={shapiro_mr_p > 0.05}")

    print(f"\nWord-order entropy distribution:")
    print(f"  mean={np.mean(wo_values):.4f}, std={np.std(wo_values):.4f}")
    print(f"  min={wo_values.min():.4f}, max={wo_values.max():.4f}")
    print(f"  Shapiro-Wilk W={shapiro_wo:.4f}, p={shapiro_wo_p:.4e}, normal={shapiro_wo_p > 0.05}")

## Visualization: Data Quality Validation Dashboard

Multi-panel summary of the six validation analyses.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Data Quality Validation Dashboard", fontsize=14, fontweight="bold")

# Panel 1: Grambank Cross-Validation scatter
ax = axes[0, 0]
gb_plot = df[df["grambank_in_overlap"] & df["ud_morph_richness"].notna()]
if len(gb_plot) >= 2:
    ax.scatter(gb_plot["ud_morph_richness"], gb_plot["grambank_morph_index"],
               c=gb_plot["feat_completeness"], cmap="RdYlGn", edgecolors="k", s=60, alpha=0.8)
    ax.set_xlabel("UD Morph Richness")
    ax.set_ylabel("Grambank Morph Index")
    for _, row in gb_plot.iterrows():
        ax.annotate(row["iso_code"], (row["ud_morph_richness"], row["grambank_morph_index"]),
                     fontsize=6, alpha=0.7)
ax.set_title("1. Grambank Cross-Validation")

# Panel 2: Entropy Threshold Sensitivity
ax = axes[0, 1]
ent_plot = df[df["entropy_threshold_10"].notna()]
if len(ent_plot) >= 2:
    ax.scatter(ent_plot["entropy_threshold_10"], ent_plot["entropy_threshold_20"],
               c="steelblue", edgecolors="k", s=60, alpha=0.8)
    lims = [min(ent_plot["entropy_threshold_10"].min(), ent_plot["entropy_threshold_20"].min()),
            max(ent_plot["entropy_threshold_10"].max(), ent_plot["entropy_threshold_20"].max())]
    ax.plot(lims, lims, "r--", alpha=0.5, label="y=x")
    ax.set_xlabel("Entropy (threshold=10)")
    ax.set_ylabel("Entropy (threshold=20)")
    ax.legend(fontsize=8)
ax.set_title("2. Entropy Threshold Sensitivity")

# Panel 3: Coverage Distribution
ax = axes[0, 2]
ax.hist(df["coverage"], bins=15, color="teal", edgecolor="k", alpha=0.7)
ax.axvline(LOW_COVERAGE_THRESHOLD, color="red", linestyle="--", label=f"Threshold={LOW_COVERAGE_THRESHOLD}")
ax.set_xlabel("Coverage (binned / total)")
ax.set_ylabel("Count")
ax.legend(fontsize=8)
ax.set_title("3. Representativeness")

# Panel 4: Autocorrelation per treebank
ax = axes[1, 0]
colors = ["#2ecc71" if not ac else "#e74c3c" for ac in df["has_autocorrelation"]]
ax.barh(range(len(df)), df["n_bins_tested"], color=colors, edgecolor="k", alpha=0.7)
ax.set_yticks(range(len(df)))
ax.set_yticklabels(df["treebank_id"], fontsize=6)
ax.set_xlabel("Bins Tested")
ax.set_title("4. Autocorrelation (red=detected)")

# Panel 5: Annotation Confound
ax = axes[1, 1]
ax.scatter(df["feat_completeness"], df["morph_richness"],
           c=df["mean_dd_all"], cmap="viridis", edgecolors="k", s=60, alpha=0.8)
ax.axvline(LOW_FC_THRESHOLD, color="red", linestyle="--", alpha=0.6, label=f"FC<{LOW_FC_THRESHOLD}")
ax.set_xlabel("Feat Completeness")
ax.set_ylabel("Morph Richness")
cb = plt.colorbar(ax.collections[0], ax=ax)
cb.set_label("Mean DD", fontsize=8)
ax.legend(fontsize=8)
ax.set_title("5. Annotation Confound")

# Panel 6: Quality Flag Distribution
ax = axes[1, 2]
quality_counts = df["overall_quality"].value_counts()
colors_q = {"GOOD": "#2ecc71", "ACCEPTABLE": "#f39c12", "CONCERN": "#e74c3c"}
bars = ax.bar(quality_counts.index, quality_counts.values,
              color=[colors_q.get(q, "gray") for q in quality_counts.index],
              edgecolor="k", alpha=0.8)
ax.set_ylabel("Count")
ax.set_title("6. Quality Distribution")
for bar, val in zip(bars, quality_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            str(val), ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("validation_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

# Print global diagnostic flags
print("\n=== Global Diagnostic Flags (from full 238-treebank analysis) ===")
for flag in global_summary.get("diagnostic_flags", []):
    print(f"  {flag}")
print(f"\nDashboard saved to validation_dashboard.png")